# W+c Analysis:

In [5]:
import hist 
import dask
import awkward as ak
import hist.dask as dhist
import dask_awkward as dak
import uproot
import numpy as np
import mplhep as mh
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import matplotlib.pyplot as plt

NanoAODSchema.warn_missing_crossrefs = False

In [6]:
events = NanoEventsFactory.from_root(
    {"../../datasets/nano106X_on_mini106X_2017_mc_NANOAOD_W1Jets_to_LNu_250K.root": "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset":"WJets"},
    mode="dask"
).events()
events

dask.awkward<from-uproot, npartitions=1>

In [10]:
# Defining the style for plots (CMS Style)
plt.style.use(mh.style.CMS)
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})

# Histogram Function:
def make_hist(data, nBins, lo, hi, xLabel, yLabel, label, fname=None, logy=False):
    histogram = dhist.Hist(h.axis.Regular(nBins, lo, hi))
    fig, ax = plt.subplots(1 ,1, figsize=(6,4))
    histogram.fill(data)
    # histogram.compute().plot(ax=ax , yerr=False, label=label)  
    mh.histplot(histogram.compute(), ax=ax, label=label, yerr=False, histtype="step") 
    mh.cms.label("Open Data", data=True, lumi=None, com=13, year=2016, loc=0)
    ax.set_xlabel(xLabel)
    ax.set_ylabel(yLabel)
    ax.legend()
    if logy:
        ax.set_yscale("log")
    if fname:
        plt.savefig(fname)
    plt.show()

## Event Selection:

Select Muons with the following cuts:
- Muon $p_{T}$ > 30 GeV (high $p_{T}$ muons).
- Muon $\eta$ < 2.4 (within the detector's acceptance).
- Muon `pfRelIso04_all` < 0.15 (highly isolated muon).
- Muon should be tight (`tightID` == 1).
- MET Transverse Mass $M_{T}$ > 50 GeV.

In [31]:
# Muon Selection Cuts: This is an object level selection cut
Muon = events.Muon
Muon_Selection_Cuts = (
    (Muon.pt > 30) &
    (abs(Muon.eta) < 2.4) &
    (Muon.tightId == 1) &
    (Muon.pfRelIso04_all < 0.15)
)
# Transverse Mass Cut: This is an object level selection cut 
MET = events.MET
Mt = np.sqrt(2 * Muon.pt * MET.pt * (1 - np.cos(Muon.phi - MET.phi)))

# Muons passing the selection cuts:
Selected_Muon = Muon[Muon_Selection_Cuts] 

# Events with only single Muon candidates and MET candidates:
EventSelection =  (ak.num(Selected_Muon) == 1)